In [1]:
import numpy as np
import cv2

# çıktı dosyalarında sıra ile değerler:
# frame, bounding_boxes(vec), feature_locations(x,y), features(vec), label


# box = xmin, xmax, ymin, ymax
def is_inside(key_point, box):
    x = key_point[0]
    y = key_point[1]

    return (x > box[0] and x < box[1] ) and (y > box[2] and y < box[3] )

# resimdeki feature için [nokta, feature, label] döner
def extract_features(image, bounding_boxes):
    sift = cv2.SIFT_create()
    key_points, descriptors = sift.detectAndCompute(image, None)
    assert len(key_points) > 0
    points = []
    feats = []
    labels = []
    for i, feat in enumerate(key_points):
        box_count = 0 #içinde olduğu box sayısı
        for box in bounding_boxes:
            if(is_inside(feat.pt, box)):
                box_count = box_count + 1
        if box_count >= 1: # araba var
            points.append(feat.pt)
            feats.append(descriptors[i])
            labels.append(1)
        elif box_count == 0: # yok
            points.append(feat.pt)
            feats.append(descriptors[i])
            labels.append(0)
        # else drop 
                
    return points, feats, labels


train_set = np.load("train_frames.npy", allow_pickle=True)
points = []
features = []
labels = []
for i in range(len(train_set)):
    row = train_set[i]
    image = cv2.imread("images/"+row[0])
    point, feat, label = extract_features(image, row[1])

    points.extend(point)
    features.extend(feat)
    labels.extend(label)


X = features
Y = labels
print(len(X), " ", len(Y))
np.save("X.npy",X,allow_pickle=True)
np.save("Y.npy",Y,allow_pickle=True)


#------------------------------------------
train_set = np.load("val_frames.npy", allow_pickle=True)
points = []
features = []
labels = []
for i in range(len(train_set)):
    row = train_set[i]
    image = cv2.imread("images/"+row[0])
    point, feat, label = extract_features(image, row[1])

    points.extend(point)
    features.extend(feat)
    labels.extend(label)


X = features
Y = labels
print(len(X), " ", len(Y))
np.save("val_X.npy",X,allow_pickle=True)
np.save("val_Y.npy",Y,allow_pickle=True)




1045065   1045065
221867   221867


In [2]:
# Windowlar için feature çıkar. Sıra ile 128 - 64 - 32 - 16
import cv2
import numpy as np

# xmin, xmax, ymin, ymax şeklinde olmalı kutular
def calculate_iou(box1, box2, threshold = 0.5):
    x1_min, x1_max, y1_min, y1_max = box1
    x2_min, x2_max, y2_min, y2_max = box2

    xmin = max(x1_min, x2_min)
    xmax = min(x1_max, x2_max)
    ymin = max(y1_min, y2_min)
    ymax = min(y1_max, y2_max)

    width = max(0, xmax- xmin)
    height = max(0, ymax- ymin)

    inter_area = width * height
    
    box1_area = (x1_max - x1_min) * (y1_max - y1_min)
    box1_area = max(0,box1_area)

    box2_area = (x2_max - x2_min) * (y2_max - y2_min)
    box2_area = max(0,box2_area)

    iou = inter_area / (box1_area + box2_area - inter_area)
    if iou <= 0:
        return False
    return iou >= threshold



def extract_window_features(image, window_size):
    nfeatures = ((window_size//16)**2) * 4
    sift = cv2.SIFT_create()  
    win_features = []
    x = 128//window_size
    x = x + (x-1)
    y = x
    for j in range(y):
        for k in range(x):

            edge_length = window_size // 2
            x_edge_start = edge_length * k
            y_edge_start = edge_length * j
            window = image[ y_edge_start : y_edge_start + window_size, x_edge_start : x_edge_start + window_size]
            kp, dc = sift.detectAndCompute(window, None)
            win_features.append(dc)

    return win_features

def extract_all_window_features(frames):
    win_sizes = [128, 64, 32, 16]

    win_features = []
    win_labels = []
    frame_length = len(frames)
    for i, win_size in enumerate(win_sizes):
        for z in range(frame_length):
            image = frames[z][0]
            image = cv2.imread("images/"+image)
            boxes = frames[z][1]
            features = extract_window_features(image, win_size)
            win_features.extend(features)
            # Window labels
            x = 128//win_size
            x = x + (x-1)
            y = x
            labels = []
            for j in range(y):
                for k in range(x):
                    found = False
                    for box in boxes:
                        edge_length = win_size // 2
                        x_min = edge_length * k
                        x_max = x_min + win_size
                        
                        y_min = edge_length * j
                        y_max = y_min + win_size

                        if calculate_iou(box, (x_min, x_max, y_min, y_max)):
                            found = True
                            break
                    if found:
                        labels.append(1)
                    else:
                        labels.append(0)
            win_labels.extend(labels)            
    
    return win_features, win_labels




val_set = np.load("val_frames.npy", allow_pickle=True)

win_features, win_labels = extract_all_window_features(val_set)

data_set = np.empty((len(win_features),2), dtype=object)
for i,(feature,label) in enumerate(zip(win_features, win_labels)):
    data_set[i,0] = feature
    data_set[i,1] = label

np.save("val_frames_with_window_features", data_set, allow_pickle=True)

#---------------------------------------------------------------------------------
test_set = np.load("test_frames.npy", allow_pickle=True)

win_features, win_labels = extract_all_window_features(test_set)

data_set = np.empty((len(win_features),2), dtype=object)
for i,(feature,label) in enumerate(zip(win_features, win_labels)):
    data_set[i,0] = feature
    data_set[i,1] = label

np.save("test_frames_with_window_features", data_set, allow_pickle=True)



